# Town-Level Dynasty Stronghold % & Curated Partitioning


In [1]:
import sys
import numpy as np
import pandas as pd
sys.path.insert(0, "..")

S = "../data/staging/"
C = "../data/curated/"
people = pd.read_parquet(S + "politicians.parquet")
terms = pd.read_parquet(S + "politician_terms.parquet")
edges = pd.read_parquet(S + "kinship_edges.parquet")
clans = pd.read_parquet(S + "clans.parquet")
memberships = pd.read_parquet(S + "hf_memberships_clean.parquet")   # used to know which towns are cities

ELECTION_YEARS = [2001, 2004, 2007, 2010, 2013, 2016, 2019, 2022, 2025]
LOCAL = ["MAYOR", "VICE MAYOR", "COUNCILOR"]
PROVINCIAL = ["GOVERNOR", "VICE GOVERNOR", "PROVINCIAL BOARD MEMBER"]
print(f"people: {len(people):,} | terms: {len(terms):,} | kinship links: {len(edges):,} | clans: {len(clans):,}")

people: 85,656 | terms: 248,685 | kinship links: 197,539 | clans: 16,379


The rule: an official is dynastic in a given year if at least one strong or medium relative held office in that year or earlier. A relative who only enters politics later doesn't count yet. So a mayor elected in 2004 whose son first wins in 2019 becomes "dynastic" only from 2019 onward, which is fair.

# Mark each official as dynastic, per election

In [2]:
info = people.set_index("person_uid")

# Strong/medium links, without same-name "twins" (probably one person under two IDs)
sm = edges[edges["tier"].isin(["strong", "medium"])]
A, B = info.loc[sm["person_a"]], info.loc[sm["person_b"]]
same_name = (A["last_key"].to_numpy() == B["last_key"].to_numpy()) & (A["first_key"].to_numpy() == B["first_key"].to_numpy())
mid_ok = A["middle_key"].isna().to_numpy() | B["middle_key"].isna().to_numpy() | (A["middle_key"].to_numpy() == B["middle_key"].to_numpy())
twin = same_name & mid_ok
sm = sm[~twin]
print(f"links skipped as possible same-person twins: {twin.sum():,}")

# For each person: the earliest year any of their relatives held office (primary terms only)
first_primary_year = terms[terms["is_primary"]].groupby("person_uid")["year"].min()
pairs = pd.concat([sm[["person_a", "person_b"]].set_axis(["person_uid", "relative"], axis=1),
                   sm[["person_b", "person_a"]].set_axis(["person_uid", "relative"], axis=1)])
pairs["relative_first_year"] = pairs["relative"].map(first_primary_year)
first_relative_year = pairs.groupby("person_uid")["relative_first_year"].min()

# One row per official per election: dynastic if a relative was in office that year or earlier
t = terms[terms["is_primary"] & terms["year"].isin(ELECTION_YEARS)].copy()
t["first_relative_year"] = t["person_uid"].map(first_relative_year)
t["is_dynastic"] = t["first_relative_year"].le(t["year"]).fillna(False).astype(bool)
t["clan_id"] = t["person_uid"].map(info["clan_id"])

t.groupby("year").agg(officials=("person_uid", "size"), source=("source", lambda s: s.mode().iloc[0]),
                      dynastic_share=("is_dynastic", "mean")).round(3)

links skipped as possible same-person twins: 3,795


,officials,source,dynastic_share
year,,,
2001,17490,openhalalan,0.163
2004,17372,hf,0.265
2007,16610,hf,0.341
2010,17507,hf,0.411
2013,17164,hf,0.449
2016,17595,hf,0.472
2019,17793,openhalalan,0.629
2022,17818,openhalalan,0.648
2025,17819,openhalalan,0.656


## Town-level stronghold %

In [3]:
# Which towns are cities (HF writes them as "CITY OF ...")
city = memberships.loc[memberships["locality"].fillna("").str.upper().str.contains("CITY"), ["province_std", "locality_std"]]
city = set(map(tuple, city.dropna().drop_duplicates().to_numpy()))

# Local officials with a known town
loc = t[t["position"].isin(LOCAL)].dropna(subset=["town_std"])
keys = ["year", "province_std", "town_std"]

# Stronghold % per town and election
town = loc.groupby(keys).agg(n_officials=("person_uid", "nunique"),
                             n_dynastic=("is_dynastic", "sum")).reset_index()
town["dynastic_share"] = (town["n_dynastic"] / town["n_officials"]).round(3)

# Is the mayor dynastic?
mayor = loc[loc["position"].eq("MAYOR")].groupby(keys).agg(mayor_uid=("person_uid", "first"),
                                                          mayor_dynastic=("is_dynastic", "max")).reset_index()
town = town.merge(mayor, on=keys, how="left")
town["mayor_dynastic"] = town["mayor_dynastic"].astype("boolean")    # empty when no mayor is recorded

# The family (clan) holding the most seats in each town
top = (loc.dropna(subset=["clan_id"]).groupby(keys + ["clan_id"])["person_uid"].nunique().rename("top_clan_seats")
          .reset_index().sort_values("top_clan_seats", ascending=False).drop_duplicates(keys)
          .rename(columns={"clan_id": "top_clan_id"}))
town = town.merge(top, on=keys, how="left")
town["top_clan_seats"] = town["top_clan_seats"].fillna(0).astype(int)
town["top_clan_share"] = (town["top_clan_seats"] / town["n_officials"]).round(3)
town["top_clan_label"] = town["top_clan_id"].map(clans.set_index("clan_id")["clan_label"])
town["fat_dynasty"] = town["top_clan_seats"] >= 2

town["is_city"] = [(p, c) in city for p, c in zip(town["province_std"], town["town_std"])]
town["stronghold_level"] = pd.cut(town["dynastic_share"], [-0.01, 0.25, 0.5, 0.75, 1.0],
                                  labels=["low (0-25%)", "moderate (25-50%)", "high (50-75%)", "stronghold (75-100%)"])
town["source"] = np.where(town["year"].between(2004, 2016), "hf", "openhalalan")

print(f"town-year rows: {len(town):,}")
display(town.groupby("year").agg(
    towns=("town_std", "size"),
    avg_dynastic_share=("dynastic_share", "mean"),
    dynastic_mayor=("mayor_dynastic", "mean"),
    fat_dynasty=("fat_dynasty", "mean"),
    strongholds=("stronghold_level", lambda s: (s == "stronghold (75-100%)").mean())).round(3))

# Towns where ONE family holds the biggest share of seats (2025)
town[town["year"].eq(2025)].sort_values(["top_clan_share", "n_officials"], ascending=False)[
    ["province_std", "town_std", "n_officials", "dynastic_share", "top_clan_label", "top_clan_seats", "top_clan_share"]].head(10)

town-year rows: 14,437


,towns,avg_dynastic_share,dynastic_mayor,fat_dynasty,strongholds
year,,,,,
2001,1605,0.148,0.31,0.313,0.002
2004,1591,0.252,0.461,0.344,0.006
2007,1513,0.328,0.563,0.364,0.018
2010,1627,0.404,0.644,0.452,0.049
2013,1593,0.445,0.674,0.485,0.065
2016,1603,0.478,0.695,0.475,0.086
2019,1634,0.620,0.816,0.534,0.271
2022,1634,0.641,0.844,0.572,0.278
2025,1637,0.648,0.842,0.600,0.315


,province_std,town_std,n_officials,dynastic_share,top_clan_label,top_clan_seats,top_clan_share
13714,MAGUINDANAO DEL SUR,AMPATUAN,10,0.700,SANGKI | MAGUINDANAO DEL SUR,7,0.700
13732,MAGUINDANAO DEL SUR,SHARIFF AGUAK,10,0.900,AMPATUAN | MAGUINDANAO DEL SUR,7,0.700
13736,MAGUINDANAO DEL SUR,TALAYAN,10,0.900,MIDTIMBANG | MAGUINDANAO DEL SUR,7,0.700
13647,LANAO DEL SUR,PICONG,10,0.600,BALINDONG | LANAO DEL SUR,6,0.600
13652,LANAO DEL SUR,TAGOLOAN II,10,0.800,CAPAL | LANAO DEL SUR,6,0.600
13715,MAGUINDANAO DEL SUR,BULUAN,10,0.700,MANGUDADATU | MAGUINDANAO DEL SUR,6,0.600
13718,MAGUINDANAO DEL SUR,DATU HOFFER AMPATUAN,10,0.700,AMPATUAN | MAGUINDANAO DEL SUR,6,0.600
13725,MAGUINDANAO DEL SUR,MAMASAPANO,9,0.889,AMPATUAN | MAGUINDANAO DEL SUR,5,0.556
12861,AKLAN,LIBACAO,10,0.700,NAVAROSA | AKLAN,5,0.500
13109,CAGAYAN,BUGUEY,10,1.000,ANTIPORDA | CAGAYAN,5,0.500


## Province-level stronghold %

In [4]:
pv = t[t["position"].isin(PROVINCIAL)].dropna(subset=["province_std"])
pk = ["year", "province_std"]

# Provincial officials (governor, vice governor, board members)
prov_tbl = pv.groupby(pk).agg(n_provincial_officials=("person_uid", "nunique"),
                              n_dynastic=("is_dynastic", "sum")).reset_index()
prov_tbl["provincial_dynastic_share"] = (prov_tbl["n_dynastic"] / prov_tbl["n_provincial_officials"]).round(3)

# Is the governor dynastic?
gov = pv[pv["position"].eq("GOVERNOR")].groupby(pk).agg(governor_uid=("person_uid", "first"),
                                                       governor_dynastic=("is_dynastic", "max")).reset_index()

# Summary of the province's towns (from Step 3)
towns_sum = town.groupby(pk).agg(n_towns=("town_std", "size"),
                                 share_towns_dynastic_mayor=("mayor_dynastic", "mean"),
                                 avg_town_dynastic_share=("dynastic_share", "mean"),
                                 n_stronghold_towns=("stronghold_level", lambda s: (s == "stronghold (75-100%)").sum())).reset_index()

prov_tbl = prov_tbl.merge(gov, on=pk, how="left").merge(towns_sum, on=pk, how="left")
prov_tbl["governor_dynastic"] = prov_tbl["governor_dynastic"].astype("boolean")
cols = ["share_towns_dynastic_mayor", "avg_town_dynastic_share"]
prov_tbl[cols] = prov_tbl[cols].astype(float).round(3)

print(f"province-year rows: {len(prov_tbl):,}")
# The 10 most dynastic provinces in 2025 (by their towns)
prov_tbl[prov_tbl["year"].eq(2025)].sort_values("avg_town_dynastic_share", ascending=False)[
    ["province_std", "governor_dynastic", "provincial_dynastic_share", "n_towns", "avg_town_dynastic_share", "n_stronghold_towns"]].head(10)

province-year rows: 720


,province_std,governor_dynastic,provincial_dynastic_share,n_towns,avg_town_dynastic_share,n_stronghold_towns
651,BILIRAN,True,0.800,8,0.800,5
692,NORTHERN SAMAR,True,0.786,24,0.763,15
686,MASBATE,True,0.857,21,0.757,11
648,BATANES,True,0.750,6,0.733,2
662,CEBU,True,0.944,53,0.725,27
659,CAPIZ,True,0.750,17,0.724,9
638,ABRA,True,1.000,27,0.722,11
656,CAMARINES NORTE,True,0.900,12,0.715,6
715,TAWI TAWI,True,0.800,11,0.709,5
661,CAVITE,True,0.833,23,0.709,12


In our README lists the PCIJ benchmark (113 of 149 cities and 71 of 82 provinces are dynastic) as a validation source. If our numbers land near PCIJ's, our method is believable. This is also one of the data quality checks for Task 10.

In [5]:
bench = pd.DataFrame({
    "cities_with_dynastic_mayor": town[town["is_city"]].groupby("year")["mayor_dynastic"].mean().astype(float),
    "provinces_with_dynastic_governor": prov_tbl.groupby("year")["governor_dynastic"].mean().astype(float)})
pcij = pd.DataFrame({"cities_with_dynastic_mayor": [113 / 149],
                     "provinces_with_dynastic_governor": [71 / 82]}, index=["PCIJ benchmark"])
(pd.concat([bench, pcij]) * 100).round(1).astype(str).add("%")

,cities_with_dynastic_mayor,provinces_with_dynastic_governor
2001,47.5%,48.7%
2004,54.0%,58.2%
2007,64.4%,60.0%
2010,67.9%,71.8%
2013,73.3%,81.2%
2016,73.8%,79.7%
2019,86.9%,79.0%
2022,89.9%,86.4%
2025,89.1%,84.1%
PCIJ benchmark,75.8%,86.6%


## Curated partitioning (one folder per election year)

In [6]:
import shutil
from pathlib import Path

C = Path("../data/curated")
officials = t[["person_uid", "year", "position", "province_std", "town_std", "party", "source",
               "is_dynastic", "first_relative_year", "clan_id"]].copy()

tables = {
    "town_dynasty_stronghold": town,
    "province_dynasty_stronghold": prov_tbl,
    "official_terms_dynastic": officials,
}
for name, df in tables.items():
    df = df.copy()
    df["year"] = df["year"].astype(int)                  # plain int so the year folders read back cleanly
    for col in df.select_dtypes("category").columns:     # e.g. stronghold_level
        df[col] = df[col].astype(str)
    out = C / name
    if out.exists():
        shutil.rmtree(out)                               # rewrite cleanly each run (idempotent)
    df.to_parquet(out, partition_cols=["year"], index=False)
    parts = sorted(p.name for p in out.iterdir())
    print(f"{name:30s} {len(df):>8,} rows -> data/curated/{name}/  ({len(parts)} folders: {parts[0]} ... {parts[-1]})")

# Check: read back ONLY 2025 (this reads just one folder)
back = pd.read_parquet(C / "town_dynasty_stronghold", filters=[("year", "=", 2025)])
print(f"\nread back year=2025 only: {len(back):,} towns")

town_dynasty_stronghold          14,437 rows -> data/curated/town_dynasty_stronghold/  (9 folders: year=2001 ... year=2025)


province_dynasty_stronghold         720 rows -> data/curated/province_dynasty_stronghold/  (9 folders: year=2001 ... year=2025)


official_terms_dynastic         157,168 rows -> data/curated/official_terms_dynastic/  (9 folders: year=2001 ... year=2025)



read back year=2025 only: 1,637 towns
